# InterUni Datathon 2026 - Final Submission Notebook

This notebook records the final modelling approach for **PURESIGMA**.

**Authors**

- Nick Liang
- Zhao Zhang
- Harshvir Singh

The competition objective is binary **log loss**, so lower scores are better. This notebook is intentionally lightweight: it documents the final pipeline, reads saved artifacts, validates the final submission file, and avoids launching any Optuna searches or model training runs.

## 1. Notebook Scope

The experimental work lives in the modelling notebooks and helper scripts. This notebook is the clean handoff version:

- load the train, test, sample submission, and saved model artifacts
- show how the first feature-engineering pass came from EDA and correlation analysis in `clean.ipynb`
- summarize the validation and public leaderboard results
- explain the final ensemble architecture
- validate the final submitted CSV
- keep code cells small, named, and easy to audit

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import log_loss, normalized_mutual_info_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.precision", 6)

In [2]:
ROOT = Path.cwd()

ID_COL = "client_id"
TARGET_COL = "default"
PREDICTION_COL = "default_probability"

TRAIN_PATH = ROOT / "train.csv"
TEST_PATH = ROOT / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "sample_submission.csv"
FINAL_SUBMISSION_PATH = ROOT / "submission_global_targeted_blend.csv"

TARGETED_CONFIG_PATH = ROOT / "global_search_targeted_best.json"
PREVIOUS_GLOBAL_CONFIG_PATH = ROOT / "global_search_best.json"

In [3]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return path


def load_json(path: Path) -> dict:
    with require_file(path).open(encoding="utf-8") as file:
        return json.load(file)


def read_submission(path: Path) -> pd.DataFrame:
    submission = pd.read_csv(require_file(path))
    required_columns = [ID_COL, PREDICTION_COL]
    missing_columns = [col for col in required_columns if col not in submission.columns]
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {missing_columns}")
    return submission[required_columns].copy()


def probability_summary(name: str, values: pd.Series | np.ndarray) -> dict:
    probs = pd.Series(values, dtype="float64")
    return {
        "name": name,
        "rows": int(probs.size),
        "mean": float(probs.mean()),
        "std": float(probs.std(ddof=0)),
        "min": float(probs.min()),
        "p05": float(probs.quantile(0.05)),
        "median": float(probs.quantile(0.50)),
        "p95": float(probs.quantile(0.95)),
        "max": float(probs.max()),
    }


def validate_submission(submission: pd.DataFrame, sample_submission: pd.DataFrame) -> dict:
    return {
        "rows_match_sample": bool(len(submission) == len(sample_submission)),
        "ids_match_sample": bool(submission[ID_COL].equals(sample_submission[ID_COL])),
        "has_missing_predictions": bool(submission[PREDICTION_COL].isna().any()),
        "all_probabilities_in_bounds": bool(submission[PREDICTION_COL].between(0.0, 1.0).all()),
        "duplicate_client_ids": int(submission[ID_COL].duplicated().sum()),
    }

## 2. EDA And Data Checks

The `clean.ipynb` pass established the basic schema before modelling. The data uses one row per client, a binary `default` target in train, and a sample-submission file that defines the required test ordering.

The key cleaning outcome was deliberately simple: no missing values were present, the provided numeric encodings were already usable for tree models, and `client_id` was treated only as an identifier.

In [4]:
train_df = pd.read_csv(require_file(TRAIN_PATH))
test_df = pd.read_csv(require_file(TEST_PATH))
sample_submission = pd.read_csv(require_file(SAMPLE_SUBMISSION_PATH))

data_summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train_df),
            "columns": train_df.shape[1],
            "missing_cells": int(train_df.isna().sum().sum()),
            "duplicate_client_ids": int(train_df[ID_COL].duplicated().sum()),
        },
        {
            "dataset": "test",
            "rows": len(test_df),
            "columns": test_df.shape[1],
            "missing_cells": int(test_df.isna().sum().sum()),
            "duplicate_client_ids": int(test_df[ID_COL].duplicated().sum()),
        },
    ]
)

target_rate = float(train_df[TARGET_COL].mean())
naive_log_loss = float(log_loss(train_df[TARGET_COL], np.repeat(target_rate, len(train_df))))

target_summary = pd.DataFrame(
    [
        {
            "target": TARGET_COL,
            "positive_rate": target_rate,
            "positive_count": int(train_df[TARGET_COL].sum()),
            "negative_count": int((1 - train_df[TARGET_COL]).sum()),
            "constant_rate_log_loss": naive_log_loss,
        }
    ]
)

display(data_summary)
display(target_summary)

,dataset,rows,columns,missing_cells,duplicate_client_ids
0,train,24000,25,0,0
1,test,6000,24,0,0


,target,positive_rate,positive_count,negative_count,constant_rate_log_loss
0,default,0.221208,5309,18691,0.528433


## 3. Correlation-Led Feature Engineering

The feature-engineering story started from the correlation analysis in `clean.ipynb`. We first created interpretable aggregate features from the raw credit-card history, then compared them with the target using two complementary signals:

## Results
- `PAY_i` columns had substantially high Spearman and NMI
- `PAY_AMTi` columns also had moderately high 
- Other predictors such as `LIMIT_BAL` and `total_pay` seemed relatively important

This suggests that these features may contain useful signal with respect to probability prediction and may be important for the models

In [5]:
PAY_STATUS_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_AMOUNT_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAYMENT_AMOUNT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]


def make_first_pass_features(df: pd.DataFrame) -> pd.DataFrame:
    """First-pass features from clean.ipynb, used to identify modelling focus areas."""
    features = df.copy()

    features["total_pay"] = features[PAYMENT_AMOUNT_COLS].sum(axis=1)
    features["total_bill"] = features[BILL_AMOUNT_COLS].sum(axis=1)

    for month, bill_col in enumerate(BILL_AMOUNT_COLS, start=1):
        features[f"credit_util_{month}"] = features[bill_col] / features["LIMIT_BAL"]

    months = np.arange(1, 7)
    month_offsets = months - months.mean()
    bill_values = features[BILL_AMOUNT_COLS].to_numpy()
    features["bill_slope"] = (bill_values * month_offsets).sum(axis=1) / (month_offsets**2).sum()

    for month in range(1, 6):
        current_bill = f"BILL_AMT{month}"
        next_bill = f"BILL_AMT{month + 1}"
        features[f"bill_abs_change_{month}_{month + 1}"] = (
            features[next_bill] - features[current_bill]
        )
        features[f"bill_pct_change_{month}_{month + 1}"] = (
            features[next_bill] - features[current_bill]
        ) / features[current_bill].replace(0, np.nan)

    features["bill_abs_change_1_6"] = features["BILL_AMT6"] - features["BILL_AMT1"]
    features["bill_pct_change_1_6"] = (
        features["BILL_AMT6"] - features["BILL_AMT1"]
    ) / features["BILL_AMT1"].replace(0, np.nan)

    features["max_delay"] = features[PAY_STATUS_COLS].max(axis=1)
    features["num_months_delayed"] = (features[PAY_STATUS_COLS] > 0).sum(axis=1)
    features["num_severe_delays"] = (features[PAY_STATUS_COLS] >= 2).sum(axis=1)
    features["ever_delayed"] = (features[PAY_STATUS_COLS] > 0).any(axis=1).astype(int)
    features["mean_pay_status"] = features[PAY_STATUS_COLS].mean(axis=1)

    return features


analysis_df = make_first_pass_features(train_df)
print(f"Original train shape: {train_df.shape}")
print(f"First-pass feature shape: {analysis_df.shape}")

Original train shape: (24000, 25)
First-pass feature shape: (24000, 51)


In [6]:
# insert correlation analysis here

,abs_spearman_with_default,nmi_with_default,mean_rank
num_severe_delays,0.390202,0.094366,1.5
ever_delayed,0.353592,0.102319,2.0
num_months_delayed,0.388466,0.089085,2.5
PAY_0,0.294178,0.050355,4.5
max_delay,0.321376,0.044595,5.0
mean_pay_status,0.258100,0.045747,5.5
PAY_2,0.217069,0.028158,8.0
PAY_3,0.198490,0.022206,9.0
PAY_5,0.164239,0.035626,9.5
PAY_4,0.175762,0.019504,10.0


# 3. Initial Feature Engineering



# 4. Initial Model Pipeline 

## Cross Validation 

Stratified 5-fold cross validation was used to maintain roughly approximate distribution across folds to maintain stability across folds

## Model Development

3 initial models were chosen
- **Baseline** - `DummyClassifier` - set to `prior` odds. Establish target to beat
- **Logistic Regression** - simple linear model to explore potential linear relations
- **XGBoost** - tree based, more flexible models to better explore non-linear dependencies and inter-feature dependnecies. Reasonable parameters were set for logistic and xgboost. 

We also considered the secondary metric for 

# 5. Initial Results and Model Selection


# 6. Extended Feature Engineering